# 04 · Feature Validation v2 — Spearman + Bootstrap + MWU

**Input**  : `data/05_features/merged_{exp_id}.parquet`  
**Output** : `data/06_validation/spearman_results.parquet`  
           + `data/06_validation/mwu_results.csv`  
           + `data/06_validation/bootstrap_ci.parquet`  
           + `outputs/figures/validation_*.png`

---

## 이 단계가 답하는 한 줄

> "ESG 언어 feature가 외부 평가(KCGS)와 어떤 방향, 어느 정도로 연결되는가?"

**왜 Spearman인가:**  
KCGS 등급은 A+/A/B+/B/C/D의 순서형(ordinal) 변수다.  
Pearson은 등간척도를 가정하므로 부적절. Spearman은 순위만 사용 → 더 적합.

**해석 기준:**  
- |ρ| < 0.10 : 무시할 수준  
- 0.10 ≤ |ρ| < 0.20 : 약한 연관 (cheap-talk 연구에서는 의미 있는 수준)  
- 0.20 ≤ |ρ| < 0.40 : 중간  

**중요:** 통계적 유의성 ≠ 실질적 의미.  
약한 음의 ρ라도 cheap-talk 가설과 *방향*이 정합하면 설명력을 갖는다.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_config import (
    D_FEATURES, D_VALID, O_FIGURES,
    RANDOM_SEED, PRIMARY_EXP, ROBUSTNESS_EXP, N_BOOTSTRAP,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(RANDOM_SEED)
print(f"Validation output dir: {D_VALID}")

Validation output dir: C:\projects\esg_dart\data\06_validation


## 1. 데이터 로드

In [2]:
merged_dfs = {}
for exp_id in ROBUSTNESS_EXP:
    # features 폴더 우선, 없으면 05_merged 폴더
    path_parquet = D_FEATURES / f"merged_{exp_id}.parquet"
    path_csv     = D_FEATURES.parent / "05_merged" / f"merged_{exp_id}.csv"
    
    if path_parquet.exists():
        df = pd.read_parquet(path_parquet)
    elif path_csv.exists():
        df = pd.read_csv(path_csv, dtype={"stock_code": str, "corp_code": str,
                                          "rcept_no": str})
        df["stock_code"] = df["stock_code"].astype(str).str.zfill(6)
        # log_tokens 추가
        if "log_tokens" not in df.columns and "total_tokens" in df.columns:
            df["log_tokens"] = np.log(df["total_tokens"] + 1)
        # kcgs 수치화
        from pipeline_config import GRADE_TO_7, GRADE_TO_4
        if "kcgs_grade_7" not in df.columns and "kcgs_grade" in df.columns:
            df["kcgs_grade_7"] = df["kcgs_grade"].map(GRADE_TO_7)
            df["kcgs_grade_4"] = df["kcgs_grade"].map(GRADE_TO_4)
            df["kcgs_grade_high"] = df["kcgs_grade"].isin(["A","A+"]).astype(int)
    else:
        print(f"[SKIP] {exp_id} — 파일 없음")
        continue
    
    # 회귀 가능 표본만 (kcgs_grade_7 유효한 행)
    df_valid = df.dropna(subset=["kcgs_grade_7"]).copy()
    merged_dfs[exp_id] = df_valid
    print(f"[{exp_id}] N={len(df_valid)} (전체 {len(df)})")

df_primary = merged_dfs[PRIMARY_EXP]
print(f"\nPRIMARY ({PRIMARY_EXP}): N={len(df_primary)}")

[exp_B] N=209 (전체 210)
[exp_E] N=209 (전체 210)
[exp_F] N=209 (전체 210)

PRIMARY (exp_F): N=209


## 2. Spearman 상관계수 — feature × KCGS grade

In [3]:
# 검정 대상 feature
FEATURE_COLS = [
    "g_signal_ratio",
    "esg_signal_ratio",
    "esg_tfidf_concentration",
    "bp_contamination_rate",
    "total_tokens",
    "log_tokens",
    "seed_tfidf_E",
    "seed_tfidf_S",
    "seed_tfidf_G",
]

TARGET = "kcgs_grade_7"

def compute_spearman(df: pd.DataFrame, features: list, target: str) -> pd.DataFrame:
    rows = []
    for feat in features:
        if feat not in df.columns:
            continue
        valid = df[[feat, target]].dropna()
        if len(valid) < 10:
            continue
        
        # variance guard
        if valid[feat].std() < 1e-8:
            rows.append({
                "feature": feat, "n": len(valid),
                "rho": None, "p_value": None,
                "significant": None, "degenerate": True,
            })
            continue
        
        rho, pval = stats.spearmanr(valid[feat], valid[target])
        rows.append({
            "feature":     feat,
            "n":           len(valid),
            "rho":         round(rho,  4),
            "p_value":     round(pval, 4),
            "significant": pval < 0.05,
            "degenerate":  False,
        })
    
    return pd.DataFrame(rows).sort_values("rho")

# Primary 결과
spearman_primary = compute_spearman(df_primary, FEATURE_COLS, TARGET)
print(f"=== Spearman ρ — {PRIMARY_EXP} ↔ KCGS grade ===")
display(spearman_primary)

=== Spearman ρ — exp_F ↔ KCGS grade ===


,feature,n,rho,p_value,significant,degenerate
0,g_signal_ratio,209,-0.1952,0.0046,True,False
2,esg_tfidf_concentration,209,-0.1748,0.0114,True,False
8,seed_tfidf_G,209,0.0126,0.8559,False,False
6,seed_tfidf_E,209,0.0445,0.5223,False,False
3,bp_contamination_rate,209,0.0500,0.4722,False,False
1,esg_signal_ratio,209,0.0832,0.2312,False,False
7,seed_tfidf_S,209,0.1978,0.0041,True,False
5,log_tokens,209,0.2796,0.0000,True,False
4,total_tokens,209,0.2796,0.0000,True,False


## 3. Bootstrap CI for ρ — primary feature

In [4]:
def bootstrap_spearman_ci(
    df: pd.DataFrame,
    feat_col: str,
    target_col: str = "kcgs_grade_7",
    n_boot: int = N_BOOTSTRAP,
    alpha: float = 0.05,
    seed: int = RANDOM_SEED,
) -> dict:
    """
    Bootstrap 재표본으로 Spearman ρ의 신뢰구간 계산.
    
    왜 Bootstrap인가:
        Spearman ρ의 분포는 ρ값에 따라 비대칭성이 달라진다.
        Normal approximation은 |ρ| > 0.5 구간에서 신뢰성이 낮다.
        Bootstrap은 분포 가정 없이 CI를 추정한다.
    """
    rng = np.random.default_rng(seed)
    valid = df[[feat_col, target_col]].dropna()
    n = len(valid)
    
    obs_rho, obs_p = stats.spearmanr(valid[feat_col], valid[target_col])
    
    boot_rhos = []
    for _ in range(n_boot):
        idx    = rng.integers(0, n, size=n)
        sample = valid.iloc[idx]
        if sample[feat_col].std() < 1e-8:
            continue
        rho_b, _ = stats.spearmanr(sample[feat_col], sample[target_col])
        boot_rhos.append(rho_b)
    
    boot_arr = np.array(boot_rhos)
    ci_lo = np.percentile(boot_arr, 100 * alpha / 2)
    ci_hi = np.percentile(boot_arr, 100 * (1 - alpha / 2))
    
    return {
        "feature":   feat_col,
        "n":         n,
        "rho_obs":   round(obs_rho, 4),
        "p_value":   round(obs_p,   4),
        "ci_lo_95":  round(ci_lo,   4),
        "ci_hi_95":  round(ci_hi,   4),
        "n_boot":    len(boot_rhos),
        "ci_excludes_zero": int(ci_lo > 0 or ci_hi < 0),
    }

print(f"Bootstrap CI 계산 중 (N_BOOT={N_BOOTSTRAP})...")
boot_rows = []
valid_feats = [f for f in FEATURE_COLS if f in df_primary.columns
               and df_primary[f].std() > 1e-8]

for feat in valid_feats:
    res = bootstrap_spearman_ci(df_primary, feat)
    boot_rows.append(res)
    print(f"  {feat:35s} ρ={res['rho_obs']:+.4f}  95%CI=[{res['ci_lo_95']:.4f}, {res['ci_hi_95']:.4f}]")

boot_df = pd.DataFrame(boot_rows)
boot_path = D_VALID / "bootstrap_ci.parquet"
boot_df.to_parquet(boot_path, index=False)
print(f"\n→ saved: {boot_path}")

Bootstrap CI 계산 중 (N_BOOT=1000)...
  g_signal_ratio                      ρ=-0.1952  95%CI=[-0.3202, -0.0568]
  esg_signal_ratio                    ρ=+0.0832  95%CI=[-0.0469, 0.2196]
  esg_tfidf_concentration             ρ=-0.1748  95%CI=[-0.2957, -0.0398]
  bp_contamination_rate               ρ=+0.0500  95%CI=[-0.0421, 0.1119]
  total_tokens                        ρ=+0.2796  95%CI=[0.1399, 0.4005]
  log_tokens                          ρ=+0.2796  95%CI=[0.1399, 0.4005]
  seed_tfidf_E                        ρ=+0.0445  95%CI=[-0.0844, 0.1707]
  seed_tfidf_S                        ρ=+0.1978  95%CI=[0.0702, 0.3259]
  seed_tfidf_G                        ρ=+0.0126  95%CI=[-0.1359, 0.1483]

→ saved: C:\projects\esg_dart\data\06_validation\bootstrap_ci.parquet


## 4. Mann-Whitney U Test — high-grade vs low-grade

In [5]:
def run_mwu_tests(
    df: pd.DataFrame,
    features: list,
    grade_col: str = "kcgs_grade_7",
    threshold: float = 5.0,   # ≥ B+
) -> pd.DataFrame:
    """
    Mann-Whitney U: high-grade (kcgs_grade_7 >= threshold) vs low-grade 그룹
    feature 분포 차이 검정.
    
    결과 해석:
        p < 0.05 → 두 그룹의 feature 분포가 통계적으로 다름
        그러나 분포가 다르다 ≠ ESG 언어가 원인. association만 보고.
    """
    df_valid = df.dropna(subset=[grade_col])
    high = df_valid[df_valid[grade_col] >= threshold]
    low  = df_valid[df_valid[grade_col] <  threshold]
    
    print(f"High-grade (≥{threshold}): N={len(high)}, Low-grade (<{threshold}): N={len(low)}")
    
    rows = []
    for feat in features:
        if feat not in df.columns:
            continue
        h_vals = high[feat].dropna()
        l_vals = low[feat].dropna()
        if len(h_vals) < 3 or len(l_vals) < 3:
            continue
        
        stat, pval = stats.mannwhitneyu(h_vals, l_vals, alternative='two-sided')
        
        # Rank-biserial correlation (effect size)
        n_h, n_l = len(h_vals), len(l_vals)
        effect_size = 1 - (2 * stat) / (n_h * n_l)
        
        rows.append({
            "feature":        feat,
            "n_high":         n_h,
            "n_low":          n_l,
            "mean_high":      round(h_vals.mean(), 5),
            "mean_low":       round(l_vals.mean(), 5),
            "mean_diff":      round(h_vals.mean() - l_vals.mean(), 5),
            "mwu_stat":       round(stat, 2),
            "p_value":        round(pval, 4),
            "effect_size_rb": round(effect_size, 4),
            "significant":    pval < 0.05,
        })
    
    return pd.DataFrame(rows).sort_values("p_value")

mwu_df = run_mwu_tests(df_primary, FEATURE_COLS)
print("\n=== Mann-Whitney U Test ===")
display(mwu_df)

mwu_path = D_VALID / "mwu_results.csv"
mwu_df.to_csv(mwu_path, index=False, encoding="utf-8-sig")
print(f"→ saved: {mwu_path}")

High-grade (≥5.0): N=186, Low-grade (<5.0): N=23

=== Mann-Whitney U Test ===


,feature,n_high,n_low,mean_high,mean_low,mean_diff,mwu_stat,p_value,effect_size_rb,significant
7,seed_tfidf_S,186,23,0.07301,0.02648,0.04654,3370.0,0.0000,-0.5755,True
4,total_tokens,186,23,5151.92473,2900.04348,2251.88125,3019.0,0.0013,-0.4114,True
5,log_tokens,186,23,8.18053,7.10150,1.07903,3019.0,0.0013,-0.4114,True
0,g_signal_ratio,186,23,0.07382,0.12610,-0.05228,1471.0,0.0147,0.3123,True
1,esg_signal_ratio,186,23,0.01281,0.00815,0.00466,2779.5,0.0193,-0.2994,True
2,esg_tfidf_concentration,186,23,0.09705,0.12964,-0.03259,1596.0,0.0474,0.2539,True
8,seed_tfidf_G,186,23,0.36721,0.40498,-0.03778,1855.0,0.3001,0.1328,False
3,bp_contamination_rate,186,23,0.00002,0.00000,0.00002,2231.0,0.3144,-0.0430,False
6,seed_tfidf_E,186,23,0.11349,0.10347,0.01001,2284.5,0.5959,-0.0680,False


→ saved: C:\projects\esg_dart\data\06_validation\mwu_results.csv


## 5. Robustness: exp별 ρ 비교 matrix

In [6]:
# 실험 × feature의 ρ matrix
rho_matrix = {}
for exp_id, df_exp in merged_dfs.items():
    rho_matrix[exp_id] = {}
    for feat in FEATURE_COLS:
        if feat not in df_exp.columns:
            continue
        valid = df_exp[[feat, TARGET]].dropna()
        if len(valid) < 5 or valid[feat].std() < 1e-8:
            rho_matrix[exp_id][feat] = None
            continue
        rho, _ = stats.spearmanr(valid[feat], valid[TARGET])
        rho_matrix[exp_id][feat] = round(rho, 4)

rho_mat_df = pd.DataFrame(rho_matrix).T
print("=== Spearman ρ Matrix (exp × feature) ===")
print(rho_mat_df.to_string())

mat_path = D_VALID / "spearman_matrix.csv"
rho_mat_df.to_csv(mat_path, encoding="utf-8-sig")
print(f"→ saved: {mat_path}")

# Δρ (exp_F - exp_B) — preprocessing 강도의 ρ 변화량
if "exp_B" in rho_matrix and "exp_F" in rho_matrix:
    delta_rows = []
    for feat in FEATURE_COLS:
        rho_B = rho_matrix.get("exp_B", {}).get(feat)
        rho_F = rho_matrix.get("exp_F", {}).get(feat)
        if rho_B is not None and rho_F is not None:
            delta_rows.append({
                "feature": feat,
                "rho_exp_B": rho_B,
                "rho_exp_F": rho_F,
                "delta_rho": round(rho_F - rho_B, 4),
            })
    delta_df = pd.DataFrame(delta_rows)
    delta_path = D_VALID / "spearman_delta_BF.csv"
    delta_df.to_csv(delta_path, index=False, encoding="utf-8-sig")
    print("\nΔρ (exp_F − exp_B):")
    display(delta_df)

=== Spearman ρ Matrix (exp × feature) ===
       g_signal_ratio  esg_signal_ratio  esg_tfidf_concentration  bp_contamination_rate  total_tokens  log_tokens  seed_tfidf_E  seed_tfidf_S  seed_tfidf_G
exp_B         -0.1967            0.0929                  -0.1854                -0.0719        0.2627      0.2627        0.0470        0.2081       -0.0216
exp_E         -0.1897            0.0762                  -0.1696                 0.0519        0.2726      0.2726        0.0410        0.2035       -0.0002
exp_F         -0.1952            0.0832                  -0.1748                 0.0500        0.2796      0.2796        0.0445        0.1978        0.0126
→ saved: C:\projects\esg_dart\data\06_validation\spearman_matrix.csv

Δρ (exp_F − exp_B):


,feature,rho_exp_B,rho_exp_F,delta_rho
0,g_signal_ratio,-0.1967,-0.1952,0.0015
1,esg_signal_ratio,0.0929,0.0832,-0.0097
2,esg_tfidf_concentration,-0.1854,-0.1748,0.0106
3,bp_contamination_rate,-0.0719,0.0500,0.1219
4,total_tokens,0.2627,0.2796,0.0169
5,log_tokens,0.2627,0.2796,0.0169
6,seed_tfidf_E,0.0470,0.0445,-0.0025
7,seed_tfidf_S,0.2081,0.1978,-0.0103
8,seed_tfidf_G,-0.0216,0.0126,0.0342


## 6. Bootstrap CI 시각화

In [7]:
fig, ax = plt.subplots(figsize=(10, 6))

boot_plot = boot_df.dropna(subset=["rho_obs"]).sort_values("rho_obs")
y_pos = range(len(boot_plot))

colors = ['crimson' if r < 0 else 'steelblue' for r in boot_plot["rho_obs"]]

ax.barh(list(y_pos), boot_plot["rho_obs"], color=colors, alpha=0.7, height=0.6)

# CI error bars
xerr_lo = boot_plot["rho_obs"] - boot_plot["ci_lo_95"]
xerr_hi = boot_plot["ci_hi_95"] - boot_plot["rho_obs"]
ax.errorbar(
    boot_plot["rho_obs"], list(y_pos),
    xerr=[xerr_lo, xerr_hi],
    fmt='none', color='black', capsize=4, linewidth=1.2
)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(boot_plot["feature"], fontsize=9)
ax.set_xlabel("Spearman ρ with KCGS grade_7")
ax.set_title(f"Feature–KCGS Spearman ρ with Bootstrap 95% CI\n({PRIMARY_EXP}, N={boot_plot['n'].iloc[0] if len(boot_plot)>0 else '?'}, B={N_BOOTSTRAP})")

neg_patch = mpatches.Patch(color='crimson', alpha=0.7, label='Negative ρ (cheap-talk consistent)')
pos_patch = mpatches.Patch(color='steelblue', alpha=0.7, label='Positive ρ')
ax.legend(handles=[neg_patch, pos_patch], loc='lower right', fontsize=8)

plt.tight_layout()
fig_path = str(O_FIGURES / f"bootstrap_ci_{PRIMARY_EXP}.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"→ figure saved: {fig_path}")

→ figure saved: C:\projects\esg_dart\outputs\figures\bootstrap_ci_exp_F.png


## 7. Spearman 결과 통합 저장

In [8]:
# 전체 실험 × feature Spearman 통합
all_spearman = []
for exp_id, df_exp in merged_dfs.items():
    sp_df = compute_spearman(df_exp, FEATURE_COLS, TARGET)
    sp_df["exp_id"] = exp_id
    all_spearman.append(sp_df)

spearman_all = pd.concat(all_spearman, ignore_index=True)
sp_path = D_VALID / "spearman_results.parquet"
spearman_all.to_parquet(sp_path, index=False)
print(f"→ saved: {sp_path}")

print("\n=== Summary ===")
print(f"Primary feature ({PRIMARY_EXP}) g_signal_ratio:")
row = boot_df[boot_df["feature"]=="g_signal_ratio"]
if len(row) > 0:
    r = row.iloc[0]
    print(f"  ρ = {r['rho_obs']:.4f}  95%CI=[{r['ci_lo_95']:.4f}, {r['ci_hi_95']:.4f}]")
    print(f"  p = {r['p_value']:.4f}")
    if r['rho_obs'] < 0:
        print("  → 음의 상관: cheap-talk 가설과 방향 정합")
    else:
        print("  → 양의 상관: cheap-talk 가설과 방향 불일치 (추가 분석 필요)")

→ saved: C:\projects\esg_dart\data\06_validation\spearman_results.parquet

=== Summary ===
Primary feature (exp_F) g_signal_ratio:
  ρ = -0.1952  95%CI=[-0.3202, -0.0568]
  p = 0.0046
  → 음의 상관: cheap-talk 가설과 방향 정합


## 8. Interpretation

**결과 해석 프레임:**

- `total_tokens` 와 KCGS grade의 ρ를 먼저 확인. 이 값이 가장 강한 신호라면  
  → verbosity bias가 분석을 지배하고 있다는 증거. 회귀에서 `log_tokens` 통제 필수.

- `g_signal_ratio`의 부호가 음(-)이면 cheap-talk 가설과 정합:  
  "KCGS 등급이 낮은 firm이 G 어휘를 더 많이 사용한다"

- `esg_tfidf_concentration`이 양(+)이면 semantic relevance 해석 가능:  
  "변별적 ESG 어휘 사용 firm이 등급이 높다"

- Bootstrap CI가 0을 포함하면 → 통계적 불확실성이 큼. 단순 ρ만 보고하지 말것.

**다음 단계:** `05_regression_v2.ipynb`  
- log_tokens, industry FE, year FE 통제 후 잔존 신호 확인  
- VIF, robust SE, Bootstrap CI for coefficients